# The best model we'll do
* bangbang



## 0. Setup


In [0]:
NOM_EQUIPE = "telecacaton"   # ← remplacez par le nom de votre équipe

# Ne touchez pas au reste
TABLE_PREDICTIONS = f"workspace.default.predictions_equipe_{NOM_EQUIPE}"
print(f"Votre table de prédictions : {TABLE_PREDICTIONS}")

In [0]:
%sql GRANT MODIFY ON TABLE workspace.default.predictions_equipe_telecacaton TO `cyprien.mas@telecom-paris.fr`;
GRANT MODIFY ON TABLE workspace.default.predictions_equipe_telecacaton TO `hugo.hennion@telecom-paris.fr`;
GRANT MODIFY ON TABLE workspace.default.predictions_equipe_telecacaton TO `clement.pesquet@telecom-paris.fr`;


## 1. First model 

### Imports & Setup

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import LongType

NOM_EQUIPE = "telecacaton"
TABLE_PREDICTIONS = f"workspace.default.predictions_equipe_{NOM_EQUIPE}"

train_df = spark.table("workspace.default.histo_ventes_train")
test_df  = spark.table("workspace.default.histo_ventes_test")

### 2.Convert to Pandas & Parse Weeks

In [0]:
# Convert to pandas for feature engineering
train_pd = train_df.toPandas()
test_pd  = test_df.toPandas()

def parse_semaine(df):
    """Split 'YYYY-WW' into integer year and week columns."""
    df = df.copy()
    df["annee"]    = df["semaine"].str.split("-").str[0].astype(int)
    df["num_sem"]  = df["semaine"].str.split("-").str[1].astype(int)
    df["week_id"]  = df["annee"] * 100 + df["num_sem"]  # sortable integer
    return df

train_pd = parse_semaine(train_pd)
test_pd  = parse_semaine(test_pd)

# Mark train/test, combine for unified feature engineering
train_pd["is_test"] = 0
test_pd["is_test"]  = 1
test_pd["quantite"] = np.nan

all_df = pd.concat([train_pd, test_pd], ignore_index=True)
all_df = all_df.sort_values(["code_agence", "code_article", "week_id"]).reset_index(drop=True)

print(f"Total rows: {len(all_df):,}  |  Train: {train_pd.shape[0]:,}  |  Test: {test_pd.shape[0]:,}")

### 3.Feature Engineering

In [0]:
def add_features(df):
    df = df.copy()
    grp = ["code_agence", "code_article"]

    # ── Lag features (shift to avoid leakage) ──────────────────────────────
    for lag in [1, 2, 4, 8, 13, 26, 52]:
        df[f"lag_{lag}"] = df.groupby(grp)["quantite"].shift(lag)

    # ── Same week last year (most powerful seasonal signal) ────────────────
    # Already captured by lag_52, but we add it explicitly for clarity
    df["lag_52_qty"] = df.groupby(grp)["quantite"].shift(52)

    # ── Rolling means (shift(1) first → no leakage) ───────────────────────
    shifted = df.groupby(grp)["quantite"].shift(1)
    for window in [4, 8, 13, 26, 52]:
        df[f"roll_mean_{window}"] = (
            shifted.groupby(df["code_agence"].astype(str) + "_" + df["code_article"].astype(str))
                   .transform(lambda x: x.rolling(window, min_periods=1).mean())
        )
        df[f"roll_std_{window}"] = (
            shifted.groupby(df["code_agence"].astype(str) + "_" + df["code_article"].astype(str))
                   .transform(lambda x: x.rolling(window, min_periods=1).std())
        )

    # ── Zero rate per pair (over all history up to that point) ────────────
    is_zero = (df["quantite"] == 0).astype(float)
    shifted_zero = is_zero.groupby(
        df["code_agence"].astype(str) + "_" + df["code_article"].astype(str)
    ).shift(1)
    df["zero_rate_26"] = (
        shifted_zero.groupby(df["code_agence"].astype(str) + "_" + df["code_article"].astype(str))
                    .transform(lambda x: x.rolling(26, min_periods=1).mean())
    )
    df["zero_rate_52"] = (
        shifted_zero.groupby(df["code_agence"].astype(str) + "_" + df["code_article"].astype(str))
                    .transform(lambda x: x.rolling(52, min_periods=1).mean())
    )

    # ── Global stats per pair (computed on full train, no leakage risk) ───
    pair_stats = (
        df[df["is_test"] == 0]
        .groupby(grp)["quantite"]
        .agg(pair_mean="mean", pair_median="median", pair_max="max", pair_count="count")
        .reset_index()
    )
    df = df.merge(pair_stats, on=grp, how="left")

    # ── Global stats per agence ───────────────────────────────────────────
    agence_stats = (
        df[df["is_test"] == 0]
        .groupby("code_agence")["quantite"]
        .agg(agence_mean="mean", agence_median="median")
        .reset_index()
    )
    df = df.merge(agence_stats, on="code_agence", how="left")

    # ── Global stats per article ──────────────────────────────────────────
    article_stats = (
        df[df["is_test"] == 0]
        .groupby("code_article")["quantite"]
        .agg(article_mean="mean", article_median="median")
        .reset_index()
    )
    df = df.merge(article_stats, on="code_article", how="left")

    # ── Seasonality encoding ──────────────────────────────────────────────
    df["sin_sem"] = np.sin(2 * np.pi * df["num_sem"] / 52)
    df["cos_sem"] = np.cos(2 * np.pi * df["num_sem"] / 52)

    return df

all_df = add_features(all_df)
print("Feature engineering done. Shape:", all_df.shape)

### 4. Train / Validation Split

In [0]:
    FEATURES = [
        # Lags
        "lag_1", "lag_2", "lag_4", "lag_8", "lag_13", "lag_26", "lag_52", "lag_52_qty",
        # Rolling stats
        "roll_mean_4", "roll_mean_8", "roll_mean_13", "roll_mean_26", "roll_mean_52",
        "roll_std_4",  "roll_std_8",  "roll_std_13",  "roll_std_26",  "roll_std_52",
        # Zero rates
        "zero_rate_26", "zero_rate_52",
        # Pair / agence / article stats
        "pair_mean", "pair_median", "pair_max", "pair_count",
        "agence_mean", "agence_median",
        "article_mean", "article_median",
        # Time
        "annee", "num_sem", "sin_sem", "cos_sem",
    ]

    TARGET = "quantite"

    # Train on pre-2025, validate on 2025-01 → 2025-26
    train_mask = (all_df["is_test"] == 0) & (all_df["semaine"] < "2025-01")
    val_mask   = (all_df["is_test"] == 0) & (all_df["semaine"].between("2025-01", "2025-26"))
    test_mask  =  all_df["is_test"] == 1

    X_train = all_df.loc[train_mask, FEATURES]
    y_train = all_df.loc[train_mask, TARGET]

    X_val   = all_df.loc[val_mask, FEATURES]
    y_val   = all_df.loc[val_mask, TARGET]

    X_test  = all_df.loc[test_mask, FEATURES]

    print(f"X_train: {X_train.shape}  |  X_val: {X_val.shape}  |  X_test: {X_test.shape}")

### 5. Train LightGBM

In [0]:
%pip install lightgbm

#dbutils.library.restartPython()

In [0]:
import lightgbm as lgb

# WAPE as custom metric for early stopping
def wape_eval(y_pred, dataset):
    y_true = dataset.get_label()
    wape = np.sum(np.abs(y_pred - y_true)) / (np.sum(y_true) + 1e-10)
    return "wape", wape, False  # lower is better

dtrain = lgb.Dataset(X_train, label=y_train)
dval   = lgb.Dataset(X_val,   label=y_val,   reference=dtrain)

params = {
    "objective":        "tweedie",   # great for zero-heavy, non-negative targets
    "tweedie_variance_power": 1.5,   # 1=Poisson-like, 2=Gamma-like; tune between 1.0–1.9
    "metric":           "None",      # we use custom WAPE
    "learning_rate":    0.05,
    "num_leaves":       127,
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     1,
    "reg_alpha":        0.1,
    "reg_lambda":       1.0,
    "n_jobs":           -1,
    "seed":             42,
    "verbose":          -1,
}

model = lgb.train(
    params,
    dtrain,
    num_boost_round       = 2000,
    valid_sets            = [dtrain, dval],
    valid_names           = ["train", "val"],
    feval                 = wape_eval,
    callbacks             = [
        lgb.early_stopping(stopping_rounds=50, min_delta=1e-4),
        lgb.log_evaluation(period=50),
    ],
)

print(f"\nBest iteration: {model.best_iteration}")

### 6.Validate & Check Score

In [0]:
val_preds_raw = model.predict(X_val, num_iteration=model.best_iteration)

# Clip negatives, round to integers
val_preds = np.clip(np.round(val_preds_raw), 0, None).astype(int)

# Compute WAPE manually
wape = np.sum(np.abs(val_preds - y_val.values)) / (np.sum(y_val.values) + 1e-10)
print(f"WAPE (validation 2025-01 → 2025-26): {wape:.4f}")
print()
print("Benchmarks:")
print("  Baseline N-1          → ~1.387")
print("  Blend N-1 + mean      → ~1.259")
print("  LightGBM target       → ~0.90–1.10")

# Feature importance (top 20)
import pandas as pd
feat_imp = pd.DataFrame({
    "feature":    FEATURES,
    "importance": model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False)
display(feat_imp.head(20))

### 7. Generate & Save Predictions

In [0]:
test_preds_raw = model.predict(X_test, num_iteration=model.best_iteration)
test_preds     = np.clip(np.round(test_preds_raw), 0, None).astype(int)

# Attach back to test rows
test_rows = all_df.loc[test_mask, ["semaine", "code_agence", "code_article"]].copy()
test_rows["quantite"] = test_preds

print(f"Predictions generated: {len(test_rows):,}  (expected 272,344)")

# Convert to Spark and save
predictions_spark = spark.createDataFrame(test_rows) \
    .withColumn("code_agence",  F.col("code_agence").cast(LongType())) \
    .withColumn("code_article", F.col("code_article").cast(LongType())) \
    .withColumn("quantite",     F.col("quantite").cast(LongType()))

predictions_spark.write.mode("overwrite").saveAsTable(TABLE_PREDICTIONS)
print(f"✅ Saved to: {TABLE_PREDICTIONS}")